# Ngày 23 · Backpropagation và autograd

- Suy ra gradient của MLP hai tầng bằng quy tắc dây chuyền và kiểm tra bằng sai phân
- Huấn luyện MLP bằng gradient descent trên dữ liệu không tách được tuyến tính
- Tự viết một autograd vô hướng nhỏ để hiểu loss.backward() làm gì

**Đọc:**
- Rumelhart, Hinton & Williams (1986), Learning Representations by Back-Propagating Errors, Nature 323, 533–536
- Goodfellow, Bengio & Courville (2016), Deep Learning, §6.5 Back-Propagation and Other Differentiation Algorithms
- Ng & Ma, CS229 Lecture Notes (2023), Chương 7 Deep learning (phần backpropagation)
- Karpathy (2020), micrograd, kho mã GitHub karpathy/micrograd

In [ ]:
import json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
from sklearn.datasets import make_moons
# make_moons: dữ liệu tổng hợp hai nửa vầng trăng (giả lập), không tách được bằng đường thẳng.
X_moon, y_moon = make_moons(300, noise=0.2, random_state=0)
def _relu(z): return np.maximum(0, z)
def _softmax(Z):
    Z = Z - Z.max(1, keepdims=True); E = np.exp(Z); return E / E.sum(1, keepdims=True)
def _forward(X, p):
    Z1 = X @ p['W1'] + p['b1']; H = _relu(Z1); return _softmax(H @ p['W2'] + p['b2']), {'X': X, 'Z1': Z1, 'H': H}
def _loss(X, y, p):
    P, _ = _forward(X, p); return float(-np.mean(np.log(P[np.arange(len(y)), y])))
_r23 = np.random.default_rng(23)
_p23 = {'W1': _r23.normal(size=(3, 6)), 'b1': _r23.normal(size=6), 'W2': _r23.normal(size=(6, 4)), 'b2': _r23.normal(size=4)}
_X23 = _r23.normal(size=(9, 3)); _y23 = _r23.integers(0, 4, 9)
def _num_grads(X, y, p, eps=1e-6):
    out = {}
    for k in p:
        g = np.zeros_like(p[k])
        for idx in np.ndindex(p[k].shape):
            a = {q: v.copy() for q, v in p.items()}; b = {q: v.copy() for q, v in p.items()}
            a[k][idx] += eps; b[k][idx] -= eps
            g[idx] = (_loss(X, y, a) - _loss(X, y, b)) / (2 * eps)
        out[k] = g
    return out


## Lan truyền ngược và huấn luyện MLP

Gợi ý: D2 = (P − onehot(y)) / n. dW2 = Hᵀ D2, db2 = tổng D2 theo hàng. DH = D2 W2ᵀ, D1 = DH ⊙ [Z1 > 0]. dW1 = Xᵀ D1, db1 = tổng D1. train_mlp: khởi tạo W ~ N(0, √(2/fan_in)) bằng np.random.default_rng(seed), b = 0, rồi lặp gradient descent toàn bộ dữ liệu.

In [ ]:
def mlp_backward(y, params, P, cache):
    # cache = {'X', 'Z1', 'H'} từ lan truyền xuôi; P là xác suất softmax. Trả dict gradient cùng khóa với params.
    raise NotImplementedError

def train_mlp(X, y, hidden, lr, steps, seed):
    # Trả (params, loss_history). Có thể dùng _forward(X, params) đã có sẵn.
    raise NotImplementedError


In [ ]:
_report('d23-backprop',
    _check('Gradient khớp sai phân trung tâm (mọi tham số)', lambda: ((lambda g, n: all(np.allclose(g[k], n[k], atol=1e-5) for k in n))(mlp_backward(_y23, _p23, *_forward(_X23, _p23)), _num_grads(_X23, _y23, _p23)), 'MLP 3 → 6 → 4 với tham số ngẫu nhiên.')),
    _check('Loss giảm và accuracy trên make_moons ≥ 0,95', lambda: ((lambda r: r[1][-1] < .5 * r[1][0] and (_forward(X_moon, r[0])[0].argmax(1) == y_moon).mean() >= .95)(train_mlp(X_moon, y_moon, 16, 0.5, 1500, 0)), 'Hồi quy logistic chỉ đạt 0,84 trên tập này.')),
)


## Autograd vô hướng tự viết

Gợi ý: Mỗi Value giữ data, grad, các nút cha và một hàm _backward. backward(): sắp xếp tô-pô các nút, đặt grad của nút cuối = 1, rồi gọi _backward theo thứ tự ngược. Luôn CỘNG DỒN vào grad (+=), không gán.

In [ ]:
class Value:
    def __init__(self, data, parents=()):
        self.data, self.grad = float(data), 0.0
        self._parents, self._backward = parents, lambda: None

    def __add__(self, other):
        raise NotImplementedError

    def __mul__(self, other):
        raise NotImplementedError

    def tanh(self):
        raise NotImplementedError

    def backward(self):
        raise NotImplementedError

    __radd__ = lambda self, other: self + other
    __rmul__ = lambda self, other: self * other


In [ ]:
def _ag1():
    a, b, c = Value(0.5), Value(-1.5), Value(0.3)
    f = (a * b + c).tanh(); f.backward()
    s = 1 - math.tanh(0.5 * -1.5 + 0.3) ** 2
    return np.allclose([a.grad, b.grad, c.grad], [s * -1.5, s * 0.5, s])
def _ag2():
    a = Value(3.0); f = a * a + a; f.backward()
    return np.isclose(f.data, 12) and np.isclose(a.grad, 7)
def _ag3():
    x = Value(2.0); f = 3 * x + 1; f.backward()
    return np.isclose(f.data, 7) and np.isclose(x.grad, 3)
_report('d23-autograd',
    _check('tanh(a·b + c): gradient khớp công thức', lambda: (_ag1(), 'd tanh(u)/du = 1 − tanh²(u).')),
    _check('Nút dùng hai lần cộng dồn gradient', lambda: (_ag2(), 'f = a² + a tại a = 3: df/da = 7.')),
    _check('Phép tính với số thường ở bên trái', lambda: (_ag3(), 'f = 3·x + 1.')),
)


## Dự án

Trong notebook cloud, thay LogisticRegression bằng MLP NumPy của bạn (784 → 128 → 10) trên 10.000 ảnh Fashion-MNIST đầu tiên. So accuracy với mốc ngày 22 và vẽ loss theo bước. Ghi lại learning rate bạn thử và chuyện gì xảy ra khi nó quá lớn.